# Pronomen-Annotation: Excel-Mappen erzeugen


| Datei | Ablage | Inhalt |
|---|---|---|
| `A3_pronomen.xlsx` | `03_annotation_laufend/pronomen/` | Haupterhebung: alle Zeilen ausfüllbar |
| `A2_pronomen_iaa.xlsx` | `03_annotation_laufend/pronomen/` | IAA-Zweitbearbeitung: **nur** die Zeilen der IAA-Teilmenge ausfüllbar |
| `A1_pronomen_iaa.xlsx` | `03_annotation_laufend/pronomen/` | IAA-Drittbearbeitung (optional, gleiche Teilmenge) |
| `modell_<name>.xlsx` | `03_annotation_laufend/pronomen/` | Modell-Referenz je Eintrag in `MODELL_NAMEN` (füllt ein Sprachmodell aus) |


**Ordnerkontext:** liegt in `00_werkzeuge/`, liest aus `02_quelldaten/`,
schreibt nach `02_quelldaten/` und `03_annotation_laufend/pronomen/`.

## Konfiguration

In [ ]:
from pathlib import Path

BASIS_QUELL  = Path("..") / "02_quelldaten"
BASIS_ARBEIT = Path("..") / "03_annotation_laufend" / "pronomen"
CSV_EIN      = BASIS_QUELL / "goldstandard_v1_pronomen.csv"

IAA_ERZAEHLUNGEN = []

HAUPTERHEBUNG = "A4"

IAA_BEARBEITER = ["A2", "A5"]

MODELL_NAMEN = ["claude-opus5", "claude-fable5", "chatgpt-gpt5.5", "copilot"]

SONDERWERTE = ["expletiv", "keine_figur", "unklar", "kein_pronomen"]
EDITIERBAR  = {"referent_gold", "kommentar"}

# Sicherheitsschalter: vorhandene Arbeitsmappen NICHT überschreiben
UEBERSCHREIBEN = False

print(f"Quelle: {CSV_EIN}")
print("IAA-Teilmenge:",
      f"{len(IAA_ERZAEHLUNGEN)} Erzählungen fest konfiguriert"
      if IAA_ERZAEHLUNGEN else "wird automatisch bestimmt")
print("Überschreiben vorhandener Arbeitsmappen:",
      "JA" if UEBERSCHREIBEN else "nein (Schutz aktiv)")
assert CSV_EIN.exists(), f"{CSV_EIN} nicht gefunden — aus 00_werkzeuge/ starten?"

Quelle: ../02_quelldaten/goldstandard_v1_pronomen.csv
IAA-Teilmenge: wird automatisch bestimmt
Überschreiben vorhandener Arbeitsmappen: nein (Schutz aktiv)


## Schritt 1 — Daten laden und IAA-Teilmenge prüfen


In [3]:
import pandas as pd

df = pd.read_csv(CSV_EIN, dtype=str, keep_default_na=False)
spalten = list(df.columns)
print(f"Geladen: {len(df)} Zeilen, {len(spalten)} Spalten")
print()

#Datenlage: alle Erzählungen mit ihrer Pronomenzahl
uebersicht = (df.groupby(["sammlung", "erzaehlung"]).size()
                .reset_index(name="n")
                .sort_values(["sammlung", "erzaehlung"]))
print("Vorhandene Erzählungen (Spalte 'erzaehlung'):")
for _, r in uebersicht.iterrows():
    print(f"  [{r['sammlung']:<10}] {r['erzaehlung']:<55} {r['n']:>3}")

#Teilmenge bestimmen: fest konfiguriert oder automatisch
if IAA_ERZAEHLUNGEN:
    unbekannt = sorted(set(IAA_ERZAEHLUNGEN) - set(df["erzaehlung"]))
    if unbekannt:
        print()
        print("FEHLER — diese konfigurierten IDs fehlen in der Datei:")
        for u in unbekannt:
            print("   ", u)
        raise ValueError(
            "Unbekannte Erzählungs-IDs (siehe Liste oben). Entweder die "
            "Schreibweise in IAA_ERZAEHLUNGEN korrigieren oder die Liste "
            "leeren, dann wird automatisch ausgewaehlt.")
    auswahl = list(IAA_ERZAEHLUNGEN)
    herkunft = "fest konfiguriert"
else:
    mittel = len(df) / uebersicht.shape[0]
    auswahl = []
    for samm in sorted(uebersicht["sammlung"].unique()):
        teil = uebersicht[uebersicht["sammlung"] == samm].copy()
        teil["abstand"] = (teil["n"] - mittel).abs()
        teil = teil.sort_values(["abstand", "erzaehlung"])   #deterministisch
        auswahl.append(teil.iloc[0]["erzaehlung"])
    herkunft = f"automatisch (Mittel {mittel:.1f} Pronomen je Erzählung)"

iaa_idx = set(df.index[df["erzaehlung"].isin(auswahl)])

print()
print(f"IAA-Teilmenge ({herkunft}):")
for e in auswahl:
    teil = df.index[df["erzaehlung"] == e]
    samm = df.loc[teil[0], "sammlung"]
    print(f"  [{samm:<10}] {e:<55} "
          f"Excel-Zeilen {teil[0]+2:>4}-{teil[-1]+2:<4} ({len(teil)})")
print(f"  Summe: {len(iaa_idx)} Einheiten = "
      f"{100*len(iaa_idx)/len(df):.0f} % von {len(df)}")
print()
print("Diese Auswahl vor Annotationsbeginn in _aufteilung.md eintragen.")


Geladen: 248 Zeilen, 12 Spalten

Vorhandene Erzählungen (Spalte 'erzaehlung'):
  [Andersen  ] 2_Das Märchen vom Sandmann                                8
  [Andersen  ] 2_Der Engel                                               9
  [Andersen  ] 2_Des Kaisers neue Kleider                               11
  [Andersen  ] 2_Die kleine Seejungfrau                                 10
  [Andersen  ] 2_Die roten Schuhe                                       17
  [Arabisch  ] 0_Alaeddin und die Wunderlampe                           21
  [Arabisch  ] 0_Die Abenteuer Sindbads des Seefahrers                   7
  [Arabisch  ] 0_Die Geschichte von Ali dem Kairenser und dem spukenden Hause in Bagdad  12
  [Arabisch  ] 0_Dschudar und seine Brüder                              15
  [Arabisch  ] 0_Khalifah der Fischer von Bagdad                         9
  [Grimm     ] 1_Der Eisenhans                                          18
  [Grimm     ] 1_Der Räuberbräutigam                                    10
  [G

## Schritt 2 — Bau-Funktion für eine Arbeitsmappe


In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Protection
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.utils import get_column_letter

BREITEN = {"sammlung": 12, "erzaehlung": 26, "tranche": 8, "block": 7,
           "satzindex": 10, "token_nr": 9, "pronomen": 12, "typ": 14,
           "satz": 55, "satz_markiert": 55, "block_markiert": 85,
           "referent_gold": 22,
           "kommentar": 30}


def baue_mappe(df, ziel, titel, hinweis, offene_zeilen=None,
               ausblenden=False):
    """Erzeugt eine Annotationsmappe.

    offene_zeilen: Menge von DataFrame-Indizes, deren Eingabezellen
                   bearbeitbar sind. None = alle Zeilen offen.
    ausblenden:    gesperrte Zeilen zusätzlich ausblenden (für A2).
    """
    alle_offen = offene_zeilen is None
    wb = Workbook()
    ws = wb.active
    ws.title = "Pronomen"

    kopf_font = Font(name="Arial", bold=True, color="FFFFFF")
    kopf_fill = PatternFill("solid", fgColor="404040")
    edit_fill = PatternFill("solid", fgColor="FFF2CC")   # gelb = ausfüllen
    grau_fill = PatternFill("solid", fgColor="D9D9D9")   # grau = gesperrt
    zell_font = Font(name="Arial")

    # Kopfzeile
    for j, name in enumerate(spalten, start=1):
        c = ws.cell(row=1, column=j, value=name)
        c.font, c.fill = kopf_font, kopf_fill
        c.alignment = Alignment(vertical="center")
        c.protection = Protection(locked=True)
        ws.column_dimensions[get_column_letter(j)].width = BREITEN.get(name, 16)

    # Datenzeilen
    for i, (idx, zeile) in enumerate(df.iterrows(), start=2):
        offen = alle_offen or idx in offene_zeilen
        for j, name in enumerate(spalten, start=1):
            c = ws.cell(row=i, column=j, value=zeile[name])
            c.font = zell_font
            if name in ("satz", "satz_markiert", "block_markiert"):
                c.alignment = Alignment(wrap_text=True, vertical="top")
            if name in EDITIERBAR:
                c.fill = edit_fill if offen else grau_fill
                c.protection = Protection(locked=not offen)
            else:
                c.protection = Protection(locked=True)
        if ausblenden and not offen:
            ws.row_dimensions[i].hidden = True

    ws.freeze_panes = "A2"
    ws.sheet_view.showGridLines = True
    letzte = get_column_letter(len(spalten))
    ws.auto_filter.ref = f"A1:{letzte}{len(df) + 1}"

    # Dropdown auf referent_gold (freie Namen bleiben erlaubt)
    rg = get_column_letter(spalten.index("referent_gold") + 1)
    dv = DataValidation(type="list",
                        formula1='"' + ",".join(SONDERWERTE) + '"',
                        allow_blank=True, showDropDown=False)
    dv.showErrorMessage = False          # Figurennamen nicht abweisen
    dv.promptTitle = "Referent"
    dv.prompt = "Figurname eintippen ODER Sonderwert aus der Liste wählen"
    ws.add_data_validation(dv)
    dv.add(f"{rg}2:{rg}{len(df) + 1}")

    # Blattschutz (invertierte Flag-Semantik)
    ws.protection.sheet = True
    ws.protection.selectUnlockedCells = False   # Eingabe möglich (WICHTIG)
    ws.protection.selectLockedCells = False     # Lesen/Kopieren erlaubt
    ws.protection.formatColumns = False         # Spaltenbreite anpassbar
    ws.protection.formatRows = False            # Zeilen ein-/ausblendbar
    ws.protection.autoFilter = False            # Filtern erlaubt
    ws.protection.sort = True                   # Sortieren verboten
    ws.protection.deleteRows = True             # Löschen verboten
    ws.protection.insertRows = True             # Einfügen verboten

    # Anleitungsblatt
    leg = wb.create_sheet("Anleitung", 0)
    zeilen = [
        (titel, ""),
        ("", ""),
        ("Auszufüllen:", "nur die GELB hinterlegten Zellen"),
        ("  referent_gold", "Figurname (wie im Märchen) ODER Sonderwert"),
        ("  kommentar", "optional, z. B. Begründung bei 'unklar'"),
        ("", ""),
        ("Umfang:", hinweis),
        ("", ""),
        ("Welches Pronomen?", "Spalte 'satz_markiert' zeigt es in **…**"),
        ("Sonderwerte:", " | ".join(SONDERWERTE)),
        ("", ""),
        ("Beispiel:", ""),
        ("  satz_markiert", "… sprang **er** auf und lief davon ."),
        ("  referent_gold", "Aladdin"),
        ("", ""),
        ("Erlaubt:", "tippen, filtern, Spaltenbreite ändern"),
        ("Nicht erlaubt:", "sortieren, Zeilen löschen/einfügen, "
                           "Blattschutz aufheben"),
        ("Volle Regeln:", "README_pronomen_annotation.md / Richtlinie §10"),
    ]
    for i, (a, b) in enumerate(zeilen, start=1):
        ca = leg.cell(row=i, column=1, value=a)
        cb = leg.cell(row=i, column=2, value=b)
        ca.font = Font(name="Arial", bold=(i == 1 or a.endswith(":")))
        cb.font = Font(name="Arial")
    leg.column_dimensions["A"].width = 22
    leg.column_dimensions["B"].width = 58
    leg.protection.sheet = True

    ziel.parent.mkdir(parents=True, exist_ok=True)
    wb.save(ziel)
    return ziel

print("Bau-Funktion definiert.")

Bau-Funktion definiert.


## Schritt 3 — Die drei Mappen erzeugen


In [ ]:
def erzeuge(ziel, **kw):
    if ziel.exists() and not UEBERSCHREIBEN:
        print(f"  [übersprungen] {ziel}  (existiert bereits — "
              f"UEBERSCHREIBEN=True setzen, um zu ersetzen)")
        return
    baue_mappe(df, ziel, **kw)
    print(f"  [geschrieben ] {ziel}")

print("Mappen:")

# 1) Haupterhebung, alle Zeilen
erzeuge(BASIS_ARBEIT / f"{HAUPTERHEBUNG}_pronomen.xlsx",
        titel=f"Pronomen-Annotation — {HAUPTERHEBUNG} (Haupterhebung)",
        hinweis=f"alle {len(df)} Zeilen ausfüllen",
        offene_zeilen=None)

# 2) Zweitbearbeitungen: nur die IAA-Teilmenge, Rest gesperrt
for nr, person in enumerate(IAA_BEARBEITER, start=2):
    erzeuge(BASIS_ARBEIT / f"{person}_pronomen_iaa.xlsx",
            titel=f"Pronomen-Annotation — {person} (unabhaengige Bearbeitung "
                  f"der IAA-Teilmenge)",
            hinweis=f"nur die {len(iaa_idx)} Zeilen der IAA-Teilmenge "
                    f"({len(auswahl)} Erzählungen); übrige Zeilen sind "
                    f"gesperrt und ausgeblendet",
            offene_zeilen=iaa_idx, ausblenden=True)

#Modell-Referenzmappen: dieselbe Teilmenge, deutlich als Maschinenreferenz
#gekennzeichnet; referent_gold fuellt spaeter das jeweilige Sprachmodell
for mname in MODELL_NAMEN:
    erzeuge(BASIS_ARBEIT / f"modell_{mname}.xlsx",
            titel=f"Pronomen — Modell-Referenz '{mname}' (keine Handannotation)",
            hinweis=(f"referent_gold fuellt das Sprachmodell '{mname}', je "
                     f"Zeile nur mit dem markierten Einzelsatz als Kontext; "
                     f"dient allein der Modellnaehe-Diagnose, nicht dem Gold"),
            offene_zeilen=iaa_idx, ausblenden=True)

Mappen:
  [geschrieben ] ../03_annotation_laufend/pronomen/A4_pronomen.xlsx
  [geschrieben ] ../03_annotation_laufend/pronomen/A2_pronomen_iaa.xlsx
  [geschrieben ] ../03_annotation_laufend/pronomen/A5_pronomen_iaa.xlsx
  [geschrieben ] ../03_annotation_laufend/pronomen/modell_claude-opus5.xlsx
  [geschrieben ] ../03_annotation_laufend/pronomen/modell_claude-fable5.xlsx
  [geschrieben ] ../03_annotation_laufend/pronomen/modell_chatgpt-gpt5.5.xlsx
  [geschrieben ] ../03_annotation_laufend/pronomen/modell_copilot.xlsx


## Schritt 4 — Kontrolle


In [6]:
from openpyxl import load_workbook

def pruefe(pfad, erwartet):
    if not pfad.exists():
        print(f"  {pfad.name:<28} (nicht vorhanden)")
        return
    ws = load_workbook(pfad)["Pronomen"]
    rg = spalten.index("referent_gold") + 1
    offen = sum(1 for i in range(2, len(df) + 2)
                if not ws.cell(row=i, column=rg).protection.locked)
    versteckt = sum(1 for i in range(2, len(df) + 2)
                    if ws.row_dimensions[i].hidden)
    status = "OK" if offen == erwartet else f"ABWEICHUNG (erwartet {erwartet})"
    print(f"  {pfad.name:<28} ausfüllbar: {offen:>3} | "
          f"ausgeblendet: {versteckt:>3} | {status}")

print("Kontrolle:")
pruefe(BASIS_ARBEIT / f"{HAUPTERHEBUNG}_pronomen.xlsx", len(df))
for person in IAA_BEARBEITER:
    pruefe(BASIS_ARBEIT / f"{person}_pronomen_iaa.xlsx", len(iaa_idx))
for mname in MODELL_NAMEN:
    pruefe(BASIS_ARBEIT / f"modell_{mname}.xlsx", len(iaa_idx))

Kontrolle:
  A4_pronomen.xlsx             ausfüllbar: 248 | ausgeblendet:   0 | OK
  A2_pronomen_iaa.xlsx         ausfüllbar:  49 | ausgeblendet: 199 | OK
  A5_pronomen_iaa.xlsx         ausfüllbar:  49 | ausgeblendet: 199 | OK
  modell_claude-opus5.xlsx     ausfüllbar:  49 | ausgeblendet: 199 | OK
  modell_claude-fable5.xlsx    ausfüllbar:  49 | ausgeblendet: 199 | OK
  modell_chatgpt-gpt5.5.xlsx   ausfüllbar:  49 | ausgeblendet: 199 | OK
  modell_copilot.xlsx          ausfüllbar:  49 | ausgeblendet: 199 | OK
